In [ ]:
import torch
import torch.nn as nn
import numpy as np

import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
p_exit, gamma = 0.4, 1.4

x_throat = 0.5  # x_length = 1.0
x_init_shock = x_throat  # random guess of shock at throat
R_in, R_throat, R_out = map(lambda x: np.sqrt(x / np.pi), (9, 1, 4))
maximum_time = 5.0
wall_delta = 0.05  # This is how close any points needs to be to be on the wall
x_close = 1e-5  # This is how close any points needs to be to be on inlet/exit/throat

In [ ]:
class DenseSubNet(nn.Module):
    def __init__(self, in_dim: int, out_dim: int, hid_dim: int, num_hidden_layers: int):
        super().__init__()
        layers = (
            [nn.Linear(in_dim, hid_dim), nn.Tanh()]
            + [
                layer
                for _ in range(num_hidden_layers)
                for layer in self.make_layer(hid_dim, hid_dim)
            ]
            + [nn.Linear(hid_dim, out_dim)]
        )

        self.net = nn.Sequential(*layers)

    @staticmethod
    def make_layer(in_dim: int, out_dim: int) -> list[nn.Module]:
        return [nn.Linear(in_dim, out_dim), nn.Tanh()]

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

    # def forward(self, xt: torch.Tensor) -> torch.Tensor:
    #     out = self.net(xt)
    #     return self.apply_pressure_linearization(xt, out)

    @staticmethod
    def apply_pressure_linearization(points: torch.Tensor, outputs: torch.Tensor):
        rho, u, p, T = (
            outputs[:, 0:1],
            outputs[:, 1:2],
            outputs[:, 2:3],
            outputs[:, 3:4],
        )
        x = points[:, 0:1]

        hard_p = x * (x - 1.0) * p + (1 - (1.0 - p_exit) * x)
        #        ^----^ 0 at in/exit, ^ 1 at inlet, linear in between, p_exit at exit

        return torch.cat([rho, u, hard_p, T], dim=1)

In [ ]:
def get_radius_at_a_distance(x: torch.Tensor) -> torch.Tensor:
    t_convergent = x / x_throat
    t_divergent = (x - x_throat) / (1.0 - x_throat)

    r_conv = R_in + (R_throat - R_in) * (3 * t_convergent**2 - 2 * t_convergent**3)
    r_div = R_throat + (R_out - R_throat) * (3 * t_divergent**2 - 2 * t_divergent**3)

    return torch.where(x < x_throat, r_conv, r_div)


def get_area(x: torch.Tensor) -> torch.Tensor:
    return torch.pi * get_radius_at_a_distance(x) ** 2


def generate_nozzle_points(n=1000, boundary_qty=0.04) -> torch.Tensor:
    interior_nums = int(n * (1 - boundary_qty))
    boundary_nums = int(n * boundary_qty) // 4  # inlet, outlet, throat, IC

    interior_x = torch.rand(interior_nums, 1, device=device, dtype=torch.float32)
    inlet_x = torch.zeros(boundary_nums, 1, device=device, dtype=torch.float32)
    outlet_x = torch.ones(boundary_nums, 1, device=device, dtype=torch.float32)
    throat_x = torch.full(
        (boundary_nums, 1), x_throat, device=device, dtype=torch.float32
    )
    ic_x = torch.rand(boundary_nums, 1, device=device, dtype=torch.float32)

    interior_t = (
        torch.rand(interior_nums, 1, device=device, dtype=torch.float32) * maximum_time
    )

    inlet_t = (
        torch.rand(boundary_nums, 1, device=device, dtype=torch.float32) * maximum_time
    )
    outlet_t = (
        torch.rand(boundary_nums, 1, device=device, dtype=torch.float32) * maximum_time
    )
    throat_t = (
        torch.rand(boundary_nums, 1, device=device, dtype=torch.float32) * maximum_time
    )

    ic_t = torch.zeros(boundary_nums, 1, device=device, dtype=torch.float32)

    all_points = torch.cat(
        [
            torch.cat([interior_x, interior_t], dim=1),
            torch.cat([inlet_x, inlet_t], dim=1),
            torch.cat([outlet_x, outlet_t], dim=1),
            torch.cat([throat_x, throat_t], dim=1),
            torch.cat([ic_x, ic_t], dim=1),
        ],
        dim=0,
    )  # shape: [n, 2]

    return all_points.requires_grad_(True)

In [ ]:
def d_(f, x):
    return torch.autograd.grad(
        f, x, grad_outputs=torch.ones_like(f), create_graph=True
    )[0]


def model_grads(model: nn.Module, xt: torch.Tensor):
    outputs = model(xt)
    rho, u, p, T = outputs[:, 0:1], outputs[:, 1:2], outputs[:, 2:3], outputs[:, 3:4]
    x, t = xt[:, 0:1], xt[:, 1:2]
    A = get_area(x)

    rho_grad, u_grad, p_grad, T_grad = d_(rho, xt), d_(u, xt), d_(p, xt), d_(T, xt)
    rho_x, rho_t = rho_grad[:, 0:1], rho_grad[:, 1:2]
    u_x, u_t = u_grad[:, 0:1], u_grad[:, 1:2]
    T_x, T_t = T_grad[:, 0:1], T_grad[:, 1:2]
    p_x = p_grad[:, 0:1]
    A_x = d_(A, x)

    return {
        "x": x,
        "t": t,
        "rho": rho,
        "u": u,
        "p": p,
        "T": T,
        "A": A,
        "rho_x": rho_x,
        "rho_t": rho_t,
        "u_x": u_x,
        "u_t": u_t,
        "T_x": T_x,
        "T_t": T_t,
        "p_x": p_x,
        "A_x": A_x,
    }

In [ ]:
def pde_residuals(gradients: dict[str, torch.Tensor]):
    rho, u, p, T, A = (
        gradients["rho"],
        gradients["u"],
        gradients["p"],
        gradients["T"],
        gradients["A"],
    )

    rho_x, u_x, p_x, T_x, A_x = (
        gradients["rho_x"],
        gradients["u_x"],
        gradients["p_x"],
        gradients["T_x"],
        gradients["A_x"],
    )

    rho_t, u_t, T_t = (
        gradients["rho_t"],
        gradients["u_t"],
        gradients["T_t"],
    )

    mass = A * rho_t + A * u * rho_x + A * rho * u_x + rho * u * A_x
    momentum = A * (gamma * rho * (u_t + u * u_x) + p_x)
    energy = rho * A * (T_t + u * T_x) + (gamma - 1) * p * (A * u_x + A_x * u)
    ideal = p - rho * T

    return (
        mass.pow(2).mean()
        + momentum.pow(2).mean()
        + energy.pow(2).mean()
        + ideal.pow(2).mean()
    )


def pre_shock_boundary_residuals(gradients: dict[str, torch.Tensor]):
    x, t, rho, u, p, T = (
        gradients["x"],
        gradients["t"],
        gradients["rho"],
        gradients["u"],
        gradients["p"],
        gradients["T"],
    )

    inlet_x_mask = x[:, 0] < x_close
    inlet_rho_residual = rho[inlet_x_mask] - 1.0
    inlet_p_residual = p[inlet_x_mask] - 1.0
    inlet_T_residual = T[inlet_x_mask] - 1.0

    initial_t_mask = t[:, 0] < x_close
    initial_rho_residual = rho[initial_t_mask] - 1.0
    initial_u_residual = u[initial_t_mask] - 0.0
    initial_p_residual = p[initial_t_mask] - 1.0
    initial_T_residual = T[initial_t_mask] - 1.0

    return (
        inlet_rho_residual.pow(2).mean()
        + inlet_p_residual.pow(2).mean()
        + inlet_T_residual.pow(2).mean()
        + initial_rho_residual.pow(2).mean()
        + initial_u_residual.pow(2).mean()
        + initial_p_residual.pow(2).mean()
        + initial_T_residual.pow(2).mean()
    )


def post_shock_boundary_residuals(gradients: dict[str, torch.Tensor]):
    x, t, rho, u, p, T = (
        gradients["x"],
        gradients["t"],
        gradients["rho"],
        gradients["u"],
        gradients["p"],
        gradients["T"],
    )

    initial_t_mask = t[:, 0] < x_close
    initial_rho_residual = rho[initial_t_mask] - 1.0
    initial_u_residual = u[initial_t_mask] - 0.0
    initial_p_residual = p[initial_t_mask] - 1.0
    initial_T_residual = T[initial_t_mask] - 1.0

    outlet_mask = x[:, 0] > 1.0 - x_close
    outlet_p_residual = p[outlet_mask] - p_exit

    return (
        outlet_p_residual.pow(2).mean()
        + initial_rho_residual.pow(2).mean()
        + initial_u_residual.pow(2).mean()
        + initial_p_residual.pow(2).mean()
        + initial_T_residual.pow(2).mean()
    )


# def boundary_residuals(gradients: dict[str, torch.Tensor]):
#     x, t, rho, u, p, T = (
#         gradients["x"],
#         gradients["t"],
#         gradients["rho"],
#         gradients["u"],
#         gradients["p"],
#         gradients["T"],
#     )

#     inlet_x_mask = x[:, 0] < x_close
#     inlet_rho_residual = rho[inlet_x_mask] - 1.0
#     inlet_p_residual = p[inlet_x_mask] - 1.0
#     inlet_T_residual = T[inlet_x_mask] - 1.0

#     initial_t_mask = t[:, 0] < x_close
#     initial_rho_residual = rho[initial_t_mask] - 1.0
#     initial_u_residual = u[initial_t_mask] - 0.0
#     initial_p_residual = p[initial_t_mask] - 1.0
#     initial_T_residual = T[initial_t_mask] - 1.0

#     outlet_mask = x[:, 0] > 1.0 - x_close
#     outlet_p_residual = p[outlet_mask] - p_exit

#     # wall boundary conditions and center flow conditions in the absence of y?

#     return (
#         inlet_rho_residual.pow(2).mean()
#         + inlet_p_residual.pow(2).mean()
#         + inlet_T_residual.pow(2).mean()
#         + initial_rho_residual.pow(2).mean()
#         + initial_u_residual.pow(2).mean()
#         + initial_p_residual.pow(2).mean()
#         + initial_T_residual.pow(2).mean()
#         + outlet_p_residual.pow(2).mean()
#     )


# def total_loss(model: nn.Module, points: torch.Tensor):
#     gradients = model_grads(model, points)
#     pde_losses = pde_residuals(gradients)
#     boundary_loss = boundary_residuals(gradients)

#     return pde_losses + 10.0 * boundary_loss, pde_losses, boundary_loss

In [ ]:
class xPINN(nn.Module):
    def __init__(
        self,
        pre_hidden_dim=128,
        post_hidden_dim=128,
        pre_hidden_layers=3,
        post_hidden_layers=3,
    ):
        super().__init__()
        # x, t -> rho, u, p, T
        self.subdomains = nn.ModuleDict(
            {
                "pre_shock": DenseSubNet(2, 4, pre_hidden_dim, pre_hidden_layers),
                "post_shock": DenseSubNet(2, 4, post_hidden_dim, post_hidden_layers),
            }
        )

        # start at the throat, per-se
        self.shock_at = nn.Parameter(
            torch.tensor([x_init_shock], device=device, dtype=torch.float32)
        )

    def forward(self, xt: torch.Tensor):  # write return type
        shock_param = torch.clamp(self.shock_at, x_throat + x_close, 1.0 - x_close)
        pre_shock_points = xt[xt[:, 0] < shock_param]
        post_shock_points = xt[xt[:, 0] >= shock_param]

        pre_shock_grads = model_grads(self.subdomains["pre_shock"], pre_shock_points)
        post_shock_grads = model_grads(self.subdomains["post_shock"], post_shock_points)

        pre_shock_pde_loss = pde_residuals(pre_shock_grads)
        pre_shock_boundary_loss = pre_shock_boundary_residuals(pre_shock_grads)
        post_shock_pde_loss = pde_residuals(post_shock_grads)
        post_shock_boundary_loss = post_shock_boundary_residuals(post_shock_grads)

        total_loss = (
            pre_shock_pde_loss
            + 10.0 * pre_shock_boundary_loss
            + post_shock_pde_loss
            + 10.0 * post_shock_boundary_loss
        )

        return {
            "total_loss": total_loss,
            "pre_shock_pde_loss": pre_shock_pde_loss,
            "pre_shock_boundary_loss": pre_shock_boundary_loss,
            "post_shock_pde_loss": post_shock_pde_loss,
            "post_shock_boundary_loss": post_shock_boundary_loss,
            "shock_at": shock_param,
        }

In [ ]:
from typing import Optional
from pathlib import Path
from datetime import datetime
from copy import deepcopy
import os


class ModelSave:
    SAVE_DIR_NAME = "saved_models_pinn"

    def __init__(self):
        file = os.path.dirname(os.path.abspath("__file__"))
        self.save_dir = Path(file).joinpath(self.SAVE_DIR_NAME)
        self.save_dir.mkdir(exist_ok=True)

    def generate_save_path(self, suffix: Optional[str] = None) -> str:
        timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
        model_name = f"hybrid-qpinn-v2-{timestamp}"
        if suffix:
            model_name += f"-{suffix}"
        return str(self.save_dir.joinpath(model_name + ".pt"))

    @staticmethod
    def snapshot(model: nn.Module):
        return deepcopy(model.state_dict())

    @staticmethod
    def restore(model: nn.Module, snapshot: dict):
        model.load_state_dict(snapshot)

    @staticmethod
    def update_best(
        model: nn.Module,
        current_loss: float,
        best_loss: Optional[float],
        best_snapshot: Optional[dict],
    ):
        if best_loss is None or current_loss < best_loss:
            return current_loss, ModelSave.snapshot(model)
        return best_loss, best_snapshot

    @staticmethod
    def save(model: nn.Module, path: str):
        model_state = ModelSave.snapshot(model)

        torch.save(model_state, path)
        print(f"Model saved to {path}")

    @staticmethod
    def load(path: str):
        if not os.path.exists(path):
            raise FileNotFoundError(f"No model found at {path}")

        model = xPINN().to(device)
        model_state = torch.load(path, map_location=device)
        ModelSave.restore(model, model_state)
        print(f"Model loaded from {path}")
        return model

In [ ]:
from typing import TypedDict


class AdamParams(TypedDict):
    epochs: int
    print_freq: int
    resample_every: int
    lr: float


def use_adam_optimizer(
    model: xPINN,
    params: AdamParams,
    history: list[dict],
    best_loss: Optional[float],
    best_state: Optional[dict],
):
    optimizer = torch.optim.Adam(model.parameters(), lr=params["lr"])

    print("starting training [Adam] optimizer...")
    for epoch in range(params["epochs"]):
        if epoch % params["resample_every"] == 0:
            points = generate_nozzle_points()

        optimizer.zero_grad()
        loss_dict = model(points)  # type: ignore
        loss = loss_dict["total_loss"]

        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        best_loss, best_state = ModelSave.update_best(
            model, loss.item(), best_loss, best_state
        )

        history.append(loss_dict)
        if (epoch + 1) % params["print_freq"] == 0:
            pde_loss = (
                loss_dict["pre_shock_pde_loss"].item()
                + loss_dict["post_shock_pde_loss"].item()
            )
            bd_loss = (
                loss_dict["pre_shock_boundary_loss"].item()
                + loss_dict["post_shock_boundary_loss"].item()
            )

            head = f"Epoch {epoch+1:<{len(str(params["epochs"]))}}/{params['epochs']}"
            print(
                f"{head} - PDE: {pde_loss:.6f}, Boundary: {bd_loss:.6f}, Total: {loss.item():.6f}"
            )

    return best_loss, best_state, history

In [ ]:
class LBFGSParams(TypedDict):
    epochs: int
    print_freq: int


def use_lbfgs_optimizer(
    model: xPINN,
    params: LBFGSParams,
    history: list[dict],
    best_loss: Optional[float],
    best_state: Optional[dict],
):
    optimizer = torch.optim.LBFGS(model.parameters(), line_search_fn="strong_wolfe")
    points = generate_nozzle_points()
    last_iter = {}

    def closure():
        optimizer.zero_grad()
        loss_dict = model(points)
        loss = loss_dict["total_loss"]

        loss.backward()
        last_iter = loss_dict.copy()
        return loss

    print("starting training [LBFGS] optimizer...")
    for epoch in range(params["epochs"]):
        loss = optimizer.step(closure)
        best_loss, best_state = ModelSave.update_best(
            model, loss.item(), best_loss, best_state
        )

        history.append(last_iter)
        if (epoch + 1) % params["print_freq"] == 0:
            pde_loss = (
                last_iter["pre_shock_pde_loss"].item()
                + last_iter["post_shock_pde_loss"].item()
            )
            bd_loss = (
                last_iter["pre_shock_boundary_loss"].item()
                + last_iter["post_shock_boundary_loss"].item()
            )

            head = f"Epoch {epoch+1:<{len(str(params['epochs']))}}/{params['epochs']}"
            print(
                f"{head} - PDE: {pde_loss:.6f}, Boundary: {bd_loss:.6f}, Total: {loss.item():.6f}"
            )

    return best_loss, best_state, history

In [ ]:
def train(
    model: xPINN,
    adam_params: AdamParams,
    lbfgs_params: LBFGSParams,
):
    best_loss, best_state = float("inf"), None
    history = []

    adam_result = use_adam_optimizer(model, adam_params, history, best_loss, best_state)
    if adam_result is None:
        raise ValueError("use_adam_optimizer returned None")

    best_loss, best_state, history = adam_result
    if best_state is not None:
        ModelSave.restore(model, best_state)
    print("[Adam] optimizer ran.", end=" ")
    ModelSave.save(model, ModelSave().generate_save_path("adam"))
    print(f"[Adam] model saved (loss: {best_loss:.6f})")

    lbfgs_result = use_lbfgs_optimizer(
        model, lbfgs_params, history, best_loss, best_state
    )
    if lbfgs_result is None:
        raise ValueError("use_lbfgs_optimizer returned None")

    best_loss, best_state, history = lbfgs_result
    if best_state is not None:
        ModelSave.restore(model, best_state)
    print("[LBFGS] optimizer ran.", end=" ")
    ModelSave.save(model, ModelSave().generate_save_path("lbfgs"))
    print(f"[LBFGS] model saved (loss: {best_loss:.6f})")

    return history

In [ ]:
model = xPINN().to(device)

{
    "total params": sum(p.numel() for p in model.parameters()),
    "trainable params": sum(p.numel() for p in model.parameters() if p.requires_grad),
}

In [ ]:
loss_history = train(
    model,
    {"epochs": 4000, "print_freq": 500, "resample_every": 100, "lr": 1e-3},
    {"epochs": 300, "print_freq": 40},
)

In [ ]:
from scipy.interpolate import interp1d
import pandas as pd


class GetReferenceData:
    def __init__(self, filename: str):
        self.reference = pd.read_csv(filename)
        self.subsonic = self.reference[self.reference["M"] < 1.0].sort_values("A")
        self.supersonic = self.reference[self.reference["M"] >= 1.0].sort_values("A")

    def get_reference(self, x_np: np.ndarray, to: str, branch_name: str):
        x_t = torch.tensor(x_np, dtype=torch.float32, device=device)[:, None]
        area_ratio = get_area(x_t).detach().cpu().numpy()
        branch = self.supersonic if branch_name == "supersonic" else self.subsonic
        return interp1d(
            branch["A"],
            self.get_branch_data(to, branch),
            bounds_error=False,
            fill_value=(branch[to].iloc[0], branch[to].iloc[-1]),  # type: ignore
        )(area_ratio).squeeze()

    @staticmethod
    def get_branch_data(to: str, branch: pd.DataFrame):
        return branch[to] if to not in ["p", "rho", "T"] else 1 / branch[to]

In [ ]:
model.eval()

x_lin  = torch.linspace(0, 1, 1000, device=device)[:, None]
t = torch.full((1000, 1), maximum_time, device=device)

eval_points = torch.cat([x_lin, t], dim=1).requires_grad_(True)
shock_location = model.shock_at.item()

with torch.enable_grad():
    pre_shock_points = eval_points[eval_points[:, 0] < shock_location]
    post_shock_points = eval_points[eval_points[:, 0] >= shock_location]

    pre_shock_out = model.subdomains["pre_shock"](pre_shock_points)
    post_shock_out = model.subdomains["post_shock"](post_shock_points)

pre_shock_points_np = pre_shock_points.detach().cpu().numpy()
post_shock_points_np = post_shock_points.detach().cpu().numpy()
pre_shock_out_np = pre_shock_out.detach().cpu().numpy()
post_shock_out_np = post_shock_out.detach().cpu().numpy()

pre_shock_rho, pre_shock_u, pre_shock_p, pre_shock_T = (
    pre_shock_out_np[:, 0],
    pre_shock_out_np[:, 1],
    pre_shock_out_np[:, 2],
    pre_shock_out_np[:, 3],
)

post_shock_rho, post_shock_u, post_shock_p, post_shock_T = (
    post_shock_out_np[:, 0],
    post_shock_out_np[:, 1],
    post_shock_out_np[:, 2],
    post_shock_out_np[:, 3],
)

pre_shock_M = np.sqrt(np.maximum((2 / (gamma - 1)) * (1 / pre_shock_T - 1), 0))
post_shock_M = np.sqrt(np.maximum((2 / (gamma - 1)) * (1 / post_shock_T - 1), 0))

In [ ]:
pde_losses = [
    h["pre_shock_pde_loss"].item() + h["post_shock_pde_loss"].item()
    for h in loss_history
]
boundary_losses = [
    h["pre_shock_boundary_loss"].item() + h["post_shock_boundary_loss"].item()
    for h in loss_history
]
total_losses = [h["total_loss"].item() for h in loss_history]

plt.figure(figsize=(10, 6))
plt.plot(pde_losses, label="PDE Loss")
plt.plot(boundary_losses, label="Boundary Loss")
plt.plot(total_losses, label="Total Loss")
plt.yscale("log")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Loss History")
plt.legend()
plt.show()

In [ ]:
from itertools import product

x = x_lin.detach().cpu().numpy().squeeze()
r = np.sqrt(get_area(torch.tensor(x, device=device)).cpu().numpy() / np.pi)

plt.figure(figsize=(12, 8))
plt.subplot(4, 1, 1)
plt.plot(x, r, label="Nozzle Radius")
plt.plot(x, -r, label="Nozzle Radius (mirrored)")
plt.title("Nozzle Shape")
plt.xlabel("x")
plt.ylabel("Radius")
plt.legend()

reference_data = GetReferenceData(
    "/kaggle/input/datasets/prantikdasiitmds/area-mach-number-ref-csv/area-mach-number-reference.csv"
)

eval_points_np = eval_points.detach().cpu().numpy()
M_ref_sup, M_ref_sub, p_ref_sup, p_ref_sub, t_ref_sup, t_ref_sub = map(
    lambda x: reference_data.get_reference(eval_points_np, x[0], x[1]),
    product(["M", "p", "T"], ["supersonic", "subsonic"]),
)

min_M, max_M = min(M_ref_sup.min(), M_ref_sub.min()), max(
    M_ref_sup.max(), M_ref_sub.max()
)
min_p, max_p = min(p_ref_sup.min(), p_ref_sub.min()), max(
    p_ref_sup.max(), p_ref_sub.max()
)
min_T, max_T = min(t_ref_sup.min(), t_ref_sub.min()), max(
    t_ref_sup.max(), t_ref_sub.max()
)

plt.subplot(4, 1, 2)
plt.plot(pre_shock_points, pre_shock_M, label="Mach Number (pre-shock)")
plt.plot(post_shock_points, post_shock_M, label="Mach Number (post-shock)")
plt.axhline(1, color="red", linestyle="--", label="Mach 1")
plt.plot(x, M_ref_sub, label="Reference (subsonic everywhere)", linestyle="dashed")
plt.plot(x, M_ref_sup, label="Reference (supersonic everywhere)", linestyle="dashed")
plt.title("Mach Number vs x")
plt.xlabel("x")
plt.ylabel("Mach Number")
plt.ylim(min_M - 0.05, max_M + 0.05)
plt.legend()

plt.subplot(4, 1, 3)
plt.plot(pre_shock_points, pre_shock_p, label="Pressure (pre-shock)")
plt.plot(post_shock_points, post_shock_p, label="Pressure (post-shock)")
plt.axhline(p_exit, color="red", linestyle="--", label="Exit Pressure")
plt.plot(x, p_ref_sub, label="Reference (subsonic everywhere)", linestyle="dashed")
plt.plot(x, p_ref_sup, label="Reference (supersonic everywhere)", linestyle="dashed")
plt.title("Pressure vs x")
plt.xlabel("x")
plt.ylabel("Pressure")
plt.ylim(min_p - 0.05, max_p + 0.05)
plt.legend()

plt.subplot(4, 1, 4)
plt.plot(x, T, label="Temperature")
plt.plot(x, t_ref_sub, label="Reference (subsonic everywhere)", linestyle="dashed")
plt.plot(x, t_ref_sup, label="Reference (supersonic everywhere)", linestyle="dashed")
plt.title("Temperature vs x")
plt.xlabel("x")
plt.ylabel("Temperature")
plt.ylim(min_T - 0.05, max_T + 0.05)
plt.legend()